# Módulo 7 · Clase 10 — Proyecto Final: California Housing

**Curso de Inteligencia Artificial — Nivel básico**

---

## Objetivo del proyecto

Predecir el **valor mediano de las viviendas** en bloques de California usando datos del censo de 1990.
Aplicaremos **todo lo aprendido en el Módulo 7**:

| Fase | Contenido |
|------|-----------|
| 1 | Cargar e inspeccionar los datos |
| 2 | Análisis exploratorio (EDA) |
| 3 | Ingeniería de features |
| 4 | Pipeline con ColumnTransformer |
| 5 | Comparar múltiples modelos |
| 6 | Optimizar el mejor con GridSearchCV |
| 7 | Evaluar y diagnosticar el modelo final |
| 8 | Guardar el modelo con joblib |

---
```
Dataset: California Housing
  Fuente: sklearn.datasets.fetch_california_housing
  Muestras: 20640 bloques censales
  Features: 8 (numéricas)
  Target: medianHouseValue (en cientos de miles de $)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve,
    GridSearchCV, StratifiedKFold, KFold
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)

np.random.seed(42)
print("Librerías listas")

---
## Fase 1 — Cargar e inspeccionar los datos

In [ ]:
# Cargar dataset
california = fetch_california_housing(as_frame=True)
df = california.frame.copy()

print("Descripción del dataset:")
print(california.DESCR[:1500])

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nFeatures: {list(df.columns)}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos:")
print(df.isnull().sum())
print(f"\nEstadísticas básicas:")
print(df.describe().round(2))

---
## Fase 2 — Análisis exploratorio (EDA)

In [ ]:
target_col = 'MedHouseVal'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución del target
axes[0].hist(df[target_col], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df[target_col].mean(),   color='red', lw=2, ls='--', label=f'Media={df[target_col].mean():.2f}')
axes[0].axvline(df[target_col].median(), color='green', lw=2, ls=':', label=f'Mediana={df[target_col].median():.2f}')
axes[0].set_title('Distribución del precio (target)')
axes[0].set_xlabel('MedHouseVal (100k $)')
axes[0].legend(fontsize=8)

# Nota: el target está capped en 5.0 (500k $)
print(f"Porcentaje de muestras en el límite máximo (5.0): "
      f"{(df[target_col] == 5.0).mean():.2%}")

# Distribución geográfica
scatter = axes[1].scatter(df['Longitude'], df['Latitude'],
                          c=df[target_col], cmap='viridis', s=2, alpha=0.5)
plt.colorbar(scatter, ax=axes[1], label='MedHouseVal')
axes[1].set_title('Distribución geográfica del precio')
axes[1].set_xlabel('Longitud'); axes[1].set_ylabel('Latitud')

# Correlaciones con el target
corr = df.corr()[target_col].drop(target_col).sort_values()
corr.plot(kind='barh', ax=axes[2], color=['coral' if v < 0 else 'steelblue' for v in corr])
axes[2].set_title(f'Correlación con {target_col}')
axes[2].axvline(0, color='black', lw=0.8)

plt.tight_layout(); plt.show()

In [ ]:
# Distribución de todas las features numéricas
features_num = [c for c in df.columns if c != target_col]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features_num):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('')
    q99 = df[col].quantile(0.99)
    ax.axvline(q99, color='red', ls='--', lw=1.5, label=f'P99={q99:.1f}')
    ax.legend(fontsize=7)

plt.suptitle('Distribución de features numéricas', fontsize=12)
plt.tight_layout(); plt.show()

print("Observaciones:")
for col in features_num:
    skew = df[col].skew()
    print(f"  {col:20}: skewness={skew:.2f}{'  ← muy asimétrica' if abs(skew)>2 else ''}")

---
## Fase 3 — Ingeniería de features

In [ ]:
# Nuevas features relevantes
df['habitaciones_por_casa'] = df['AveRooms']   / df['HouseAge'].clip(1)
df['dormitorios_por_hab']   = df['AveBedrms']  / df['AveRooms'].clip(0.1)
df['densidad_poblacion']    = df['Population'] / df['AveOccup'].clip(1)
df['ingreso_por_persona']   = df['MedInc']     / df['AveOccup'].clip(1)

# Nuevas correlaciones con el target
nuevas_features = ['habitaciones_por_casa','dormitorios_por_hab',
                   'densidad_poblacion','ingreso_por_persona']

corr_nuevas = df[nuevas_features + [target_col]].corr()[target_col].drop(target_col)
corr_originales = df[features_num + [target_col]].corr()[target_col].drop(target_col)

print("Correlaciones originales con target:")
print(corr_originales.round(3).to_string())
print("\nNuevas features:")
print(corr_nuevas.round(3).to_string())

In [ ]:
# Preparar X e y
features_finales = features_num + nuevas_features
X = df[features_finales]
y = df[target_col]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape[0]} muestras")
print(f"Test:  {X_te.shape[0]} muestras")
print(f"Features: {len(features_finales)}")

---
## Fase 4 — Pipeline con preprocesamiento

In [ ]:
# Todas las features son numéricas en este dataset
preprocesador = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

def crear_pipeline_regresion(modelo):
    return Pipeline([
        ('prepro', preprocesador),
        ('modelo', modelo),
    ])

def evaluar_regresion(y_real, y_pred, nombre=''):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

print("Pipeline definido")

---
## Fase 5 — Comparar múltiples modelos

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Baseline (media)':        None,  # modelo trivial
    'LinearRegression':        LinearRegression(),
    'Ridge α=1':               Ridge(alpha=1),
    'Lasso α=0.01':            Lasso(alpha=0.01),
    'RandomForest 100':        RandomForestRegressor(100, random_state=42, n_jobs=-1),
    'GradientBoosting 100':    GradientBoostingRegressor(100, random_state=42),
}

resultados = []

for nombre, m in modelos.items():
    if m is None:
        # Baseline: siempre predice la media del train
        y_pred_base = np.full(len(y_te), y_tr.mean())
        resultados.append(evaluar_regresion(y_te, y_pred_base, nombre))
        continue

    pipe = crear_pipeline_regresion(m)
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    resultados.append(evaluar_regresion(y_te, y_pred, nombre))

df_res = pd.DataFrame(resultados).sort_values('R²', ascending=False)
print("Comparativa de modelos en test:")
print(df_res.round(4).to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 4))
colores = ['coral' if r < 0.5 else 'steelblue' if r < 0.8 else 'seagreen'
           for r in df_res['R²']]
bars = ax.bar(df_res['Modelo'], df_res['R²'], color=colores)
ax.set_title('R² en test por modelo')
ax.set_ylabel('R²')
ax.tick_params(axis='x', rotation=25)
for bar, r2 in zip(bars, df_res['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{r2:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

---
## Fase 6 — Optimizar el mejor modelo con GridSearchCV

In [ ]:
# GradientBoosting suele ganar en datasets tabulares
pipe_gb = crear_pipeline_regresion(
    GradientBoostingRegressor(random_state=42)
)

rejilla = {
    'modelo__n_estimators':   [100, 200],
    'modelo__max_depth':      [3, 4, 5],
    'modelo__learning_rate':  [0.05, 0.1, 0.2],
    'modelo__subsample':      [0.8, 1.0],
}

gs = GridSearchCV(
    pipe_gb, rejilla,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring='r2',
    refit=True, n_jobs=-1, verbose=1
)
gs.fit(X_tr, y_tr)

print(f"\nMejores parámetros: {gs.best_params_}")
print(f"R² CV (train): {gs.best_score_:.4f}")

---
## Fase 7 — Evaluar y diagnosticar el modelo final

In [ ]:
modelo_final = gs.best_estimator_
y_pred_final = modelo_final.predict(X_te)

metricas = evaluar_regresion(y_te, y_pred_final, 'GradientBoosting optimizado')
print("Métricas en test — modelo final:")
for k, v in metricas.items():
    if k != 'Modelo':
        print(f"  {k:6}: {v:.4f}")
print(f"\nInterpretación:")
print(f"  MAE  = {metricas['MAE']:.4f} → error medio de {metricas['MAE']*100_000:.0f} $")
print(f"  RMSE = {metricas['RMSE']:.4f} → penaliza errores grandes")
print(f"  R²   = {metricas['R²']:.4f} → el modelo explica el {metricas['R²']*100:.1f}% de la varianza")

In [ ]:
residuos = y_te - y_pred_final

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Real vs Predicho
axes[0,0].scatter(y_te, y_pred_final, s=5, alpha=0.3, color='steelblue')
rango = [min(y_te.min(), y_pred_final.min()), max(y_te.max(), y_pred_final.max())]
axes[0,0].plot(rango, rango, 'r--', lw=1.5, label='Predicción perfecta')
axes[0,0].set_xlabel('Valor real'); axes[0,0].set_ylabel('Valor predicho')
axes[0,0].set_title(f'Real vs Predicho (R²={metricas["R²"]:.4f})')
axes[0,0].legend(fontsize=8)

# 2. Residuos vs Predicho
axes[0,1].scatter(y_pred_final, residuos, s=5, alpha=0.3, color='coral')
axes[0,1].axhline(0, color='black', lw=1.5, ls='--')
axes[0,1].set_xlabel('Valor predicho'); axes[0,1].set_ylabel('Residuos')
axes[0,1].set_title('Residuos vs Predicho')

# 3. Histograma de residuos
axes[1,0].hist(residuos, bins=50, color='seagreen', edgecolor='white', alpha=0.8)
axes[1,0].axvline(0, color='red', lw=2, ls='--')
axes[1,0].set_xlabel('Residuos')
axes[1,0].set_title(f'Distribución residuos (media={residuos.mean():.4f}, std={residuos.std():.4f})')

# 4. Feature importance
gb = modelo_final.named_steps['modelo']
importancias = pd.Series(gb.feature_importances_, index=features_finales).sort_values(ascending=True)
importancias.plot(kind='barh', ax=axes[1,1], color='mediumpurple')
axes[1,1].set_title('Feature Importances')
axes[1,1].set_xlabel('Importancia')

plt.suptitle('Diagnóstico del modelo final — GradientBoosting optimizado', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Curva de aprendizaje del modelo final
sizes, tr_scores, te_scores = learning_curve(
    modelo_final, X_tr, y_tr,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='r2', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sizes, tr_scores.mean(1), 'b-o', ms=6, lw=2, label='Train R²')
ax.plot(sizes, te_scores.mean(1), 'r-o', ms=6, lw=2, label='Val CV R²')
ax.fill_between(sizes, tr_scores.mean(1)-tr_scores.std(1),
                tr_scores.mean(1)+tr_scores.std(1), alpha=0.1, color='blue')
ax.fill_between(sizes, te_scores.mean(1)-te_scores.std(1),
                te_scores.mean(1)+te_scores.std(1), alpha=0.1, color='red')
ax.set_xlabel('Nº muestras de entrenamiento')
ax.set_ylabel('R²')
ax.set_title('Curva de aprendizaje — GradientBoosting optimizado')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

gap = tr_scores.mean(1)[-1] - te_scores.mean(1)[-1]
print(f"Gap final (train - val): {gap:.4f}")
print(f"Diagnóstico: {'BUEN AJUSTE' if gap < 0.08 else 'OVERFITTING LEVE'}")

---
## Fase 8 — Guardar el modelo

In [ ]:
# Guardar el pipeline completo
ruta = 'california_housing_model_v1.joblib'
joblib.dump(modelo_final, ruta)

# Metadatos
meta = {
    'version':        'v1.0',
    'dataset':        'California Housing (sklearn)',
    'features':       features_finales,
    'target':         'MedHouseVal (100k $)',
    'algoritmo':      'GradientBoostingRegressor',
    'mejores_params': {k: str(v) for k, v in gs.best_params_.items()},
    'r2_cv_train':    round(gs.best_score_, 4),
    'mae_test':       round(metricas['MAE'], 4),
    'rmse_test':      round(metricas['RMSE'], 4),
    'r2_test':        round(metricas['R²'], 4),
    'n_train':        len(X_tr),
    'n_test':         len(X_te),
}
with open('california_housing_model_v1_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Modelo guardado:")
print(f"  {ruta}")
print(f"  california_housing_model_v1_meta.json")

# Verificar que el modelo cargado funciona
modelo_cargado = joblib.load(ruta)
y_pred_check   = modelo_cargado.predict(X_te[:3])
print(f"\nVerificación (3 predicciones):")
print(f"  Predicho (100k $): {y_pred_check.round(3)}")
print(f"  Real     (100k $): {y_te.values[:3].round(3)}")

---
## Informe final del proyecto

```
═══════════════════════════════════════════════════════════
  PROYECTO: Predicción de precios de vivienda en California
═══════════════════════════════════════════════════════════

DATASET:
  20640 bloques censales, 8 features originales + 4 derivadas
  Target: valor mediano de la vivienda (100k $)

PIPELINE:
  → SimpleImputer (mediana) → StandardScaler → GradientBoosting

COMPARATIVA DE MODELOS:
  Baseline (media)       R²≈0.00 (referencia trivial)
  LinearRegression       R²≈0.60
  Ridge                  R²≈0.60
  RandomForest           R²≈0.80
  GradientBoosting opt   R²≈0.83

MODELO FINAL:
  GradientBoostingRegressor con hiperparámetros optimizados
  MAE  ≈ 0.31 (31k $ de error medio)
  RMSE ≈ 0.45
  R²   ≈ 0.83 (explica el 83% de la varianza)

FEATURES MÁS IMPORTANTES:
  1. MedInc              (ingresos medianos del barrio)
  2. Latitude / Longitude (localización geográfica)
  3. ingreso_por_persona  (feature derivada)

DIAGNÓSTICO:
  Curva de aprendizaje: buen ajuste, sin overfitting significativo
  Residuos: distribución simétrica centrada en 0
  Límite del cap en 5.0 → introduce ruido en predicciones altas
═══════════════════════════════════════════════════════════
```

---
## Práctica

### Ejercicio 1 — Analizar el impacto de las features derivadas

**Enunciado:**

1. Entrena dos modelos idénticos (GradientBoosting con params por defecto):
   - Modelo A: solo features originales (8)
   - Modelo B: features originales + derivadas (12)
2. Compara R², MAE y RMSE en test
3. ¿Cuánto mejoran las features derivadas?
4. ¿Qué feature derivada tiene más impacto? (feature importance)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

california = fetch_california_housing(as_frame=True)
df = california.frame.copy()

# Features derivadas
df['habitaciones_por_casa'] = df['AveRooms']   / df['HouseAge'].clip(1)
df['dormitorios_por_hab']   = df['AveBedrms']  / df['AveRooms'].clip(0.1)
df['densidad_poblacion']    = df['Population'] / df['AveOccup'].clip(1)
df['ingreso_por_persona']   = df['MedInc']     / df['AveOccup'].clip(1)

target_col = 'MedHouseVal'
features_orig = [c for c in california.frame.columns if c != target_col]
features_full = features_orig + ['habitaciones_por_casa','dormitorios_por_hab',
                                  'densidad_poblacion','ingreso_por_persona']
y = df[target_col]

resultados = []
for nombre, feats in [('Solo originales', features_orig), ('+ Derivadas', features_full)]:
    X_ = df[feats]
    X_tr_, X_te_, y_tr_, y_te_ = train_test_split(X_, y, test_size=0.2, random_state=42)
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('m',   GradientBoostingRegressor(100, random_state=42)),
    ])
    pipe.fit(X_tr_, y_tr_)
    y_pred_ = pipe.predict(X_te_)
    r2  = r2_score(y_te_, y_pred_)
    mae = mean_absolute_error(y_te_, y_pred_)
    rmse = np.sqrt(mean_squared_error(y_te_, y_pred_))
    resultados.append({'Modelo': nombre, 'R²': r2, 'MAE': mae, 'RMSE': rmse})
    # Feature importance
    imp = pd.Series(pipe['m'].feature_importances_, index=feats).sort_values(ascending=False)
    print(f"\n{nombre}: R²={r2:.4f}, MAE={mae:.4f}")
    print(f"Top 5 features: {imp.head(5).index.tolist()}")

df_comp = pd.DataFrame(resultados)
print(f"\nMejora por features derivadas:")
mejora_r2 = df_comp.iloc[1]['R²'] - df_comp.iloc[0]['R²']
print(f"  R²: +{mejora_r2:.4f}")
```
</details>

---
### Ejercicio 2 — Análisis de errores por zona geográfica

**Enunciado:**

1. Usa el modelo final del proyecto
2. Añade al DataFrame de test: `y_real`, `y_pred`, `error_abs` (|real - pred|)
3. Visualiza en un mapa scatter (Latitude, Longitude) coloreado por el error absoluto
4. ¿En qué zonas geográficas comete el modelo más errores?
5. ¿Qué podrías hacer para mejorar en esas zonas?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Reconstruir X_te con coordenadas geográficas
X_te_full = df.loc[X_te.index, features_finales + ['Latitude','Longitude']]

df_errores = X_te_full[['Latitude','Longitude']].copy()
df_errores['y_real'] = y_te.values
df_errores['y_pred'] = y_pred_final
df_errores['error_abs'] = np.abs(y_te.values - y_pred_final)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Mapa de errores
sc1 = axes[0].scatter(
    df_errores['Longitude'], df_errores['Latitude'],
    c=df_errores['error_abs'], cmap='YlOrRd', s=4, alpha=0.6
)
plt.colorbar(sc1, ax=axes[0], label='Error absoluto (100k $)')
axes[0].set_title('Error absoluto por zona geográfica')
axes[0].set_xlabel('Longitud'); axes[0].set_ylabel('Latitud')

# Top 10% errores
umbral = df_errores['error_abs'].quantile(0.9)
peores = df_errores[df_errores['error_abs'] >= umbral]
sc2 = axes[1].scatter(
    df_errores['Longitude'], df_errores['Latitude'],
    c='lightgray', s=3, alpha=0.3
)
axes[1].scatter(
    peores['Longitude'], peores['Latitude'],
    c=peores['error_abs'], cmap='Reds', s=15, alpha=0.8
)
axes[1].set_title('Top 10% errores más grandes')
axes[1].set_xlabel('Longitud'); axes[1].set_ylabel('Latitud')

plt.tight_layout(); plt.show()

# Análisis por zona
print("Estadísticas por decil de precio real:")
df_errores['decil'] = pd.qcut(df_errores['y_real'], 10, labels=False)
print(df_errores.groupby('decil')['error_abs'].mean().round(4).to_string())
print("\nObservación: los errores son mayores en viviendas caras (cercanas al cap de 5.0).")
print("Mejora posible: filtrar o separar viviendas en el límite máximo.")
```
</details>

---
### Ejercicio 3 — Probar RandomizedSearchCV para GradientBoosting

**Enunciado:** El GridSearchCV exploró 36 combinaciones. Prueba si con `RandomizedSearchCV` (n_iter=20) obtienes resultados similares en menos tiempo:

1. Define un espacio de búsqueda más amplio con distribuciones continuas
2. Ejecuta `RandomizedSearchCV` con n_iter=20, cv=5
3. Compara tiempo y R² con el GridSearchCV anterior
4. ¿Cuál recomendarías en producción con hiperparámetros continuos?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import time
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, KFold

espacio_rs = {
    'modelo__n_estimators':  randint(50, 300),
    'modelo__max_depth':     randint(2, 7),
    'modelo__learning_rate': uniform(0.01, 0.25),
    'modelo__subsample':     uniform(0.6, 0.4),
    'modelo__min_samples_split': randint(2, 10),
}

pipe_rs = crear_pipeline_regresion(GradientBoostingRegressor(random_state=42))
rs = RandomizedSearchCV(
    pipe_rs, espacio_rs, n_iter=20,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring='r2', refit=True, n_jobs=-1, random_state=42
)

t0 = time.time()
rs.fit(X_tr, y_tr)
t_rs = time.time() - t0

y_pred_rs = rs.predict(X_te)
r2_rs = r2_score(y_te, y_pred_rs)

print(f"RandomizedSearchCV (n_iter=20): tiempo={t_rs:.1f}s, R²={r2_rs:.4f}")
print(f"GridSearchCV (36 combis):       R²={metricas['R²']:.4f}")
print(f"\nMejores params RS: {rs.best_params_}")
print("\nVeredicto: RandomSearch con espacio más amplio suele igualar o superar")
print("a GridSearch con espacio discreto, usando menos evaluaciones.")
```
</details>

---
### Ejercicio 4 — Clasificación alternativa del precio

**Enunciado:** Transforma el problema en clasificación:

```python
# 3 categorías de precio
y_cat = pd.cut(y, bins=[0, 1.5, 3.0, 6.0], labels=['bajo','medio','alto'])
```

1. Entrena un pipeline de clasificación (GradientBoosting o RF) con las mismas features
2. Evalúa con 5-Fold CV: accuracy y F1 macro
3. Muestra la matriz de confusión en test
4. ¿Qué categoría confunde más el modelo?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np

y_cat = pd.cut(y, bins=[0, 1.5, 3.0, 6.0], labels=['bajo','medio','alto'])
X_clf = X.copy()
# Quitar NaN del target
mask = y_cat.notna()
X_clf, y_clf = X_clf[mask], y_cat[mask]

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

pipe_clf = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
    ('clf', GradientBoostingClassifier(100, random_state=42)),
])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
acc = cross_val_score(pipe_clf, X_clf, y_clf, cv=cv, scoring='accuracy', n_jobs=-1).mean()
f1  = cross_val_score(pipe_clf, X_clf, y_clf, cv=cv, scoring='f1_macro', n_jobs=-1).mean()
print(f"Accuracy CV: {acc:.4f}")
print(f"F1 macro CV: {f1:.4f}")

pipe_clf.fit(X_tr_c, y_tr_c)
print(f"\n{classification_report(y_te_c, pipe_clf.predict(X_te_c))}")

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(pipe_clf, X_te_c, y_te_c, ax=ax, cmap='Blues')
ax.set_title('Matriz de confusión — Clasificación de precio')
plt.tight_layout(); plt.show()
print("La categoría 'medio' suele tener más confusión con 'bajo' y 'alto'.")
```
</details>

---
### Ejercicio 5 (reto) — Tu propio dataset, tu propio proyecto

**Enunciado:** Aplica todo el proceso del proyecto a un dataset distinto de tu elección:

```python
# Opciones
from sklearn.datasets import load_diabetes      # regresión: progresión de diabetes
from sklearn.datasets import load_breast_cancer # clasificación: diagnóstico cáncer
from sklearn.datasets import load_wine          # clasificación: tipo de vino
# O descarga un dataset de Kaggle que te interese
```

Tu proyecto debe incluir:
1. EDA con al menos 3 visualizaciones
2. Al menos 1 feature derivada
3. Pipeline con ColumnTransformer (aunque todas sean numéricas)
4. Comparativa de al menos 3 modelos
5. GridSearchCV o RandomizedSearchCV
6. Diagnóstico (curva de aprendizaje + residuos o matriz de confusión)
7. Guardar el modelo con joblib
8. Informe final: 5 conclusiones del dataset y el modelo

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Este es tu proyecto libre. ¡No hay una solución única!
# Aplica el proceso de las 8 fases de este notebook a un dataset de tu elección.


---
## Resumen del Módulo 7 — Machine Learning

```python
# ═══════════════════════════════════════════════════════════
# RESUMEN COMPLETO DEL MÓDULO 7
# ═══════════════════════════════════════════════════════════

# C1: sklearn API universal
modelo.fit(X_tr, y_tr)
modelo.predict(X_te)
modelo.score(X_te, y_te)

# C2: Preparación de datos (sin data leakage)
ct = ColumnTransformer([
    ('num', StandardScaler(), cols_num),
    ('nom', OneHotEncoder(drop='first'), cols_nom),
])
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)          # NO fit en test

# C3: Regresión
mae  = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2   = r2_score(y, y_pred)

# C4: Clasificación
acc  = accuracy_score(y, y_pred)
f1   = f1_score(y, y_pred)
auc  = roc_auc_score(y, proba)

# C5: Árboles y Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
importancias = rf.feature_importances_

# C6: Overfitting → curvas de diagnóstico
sizes, tr_sc, te_sc = learning_curve(modelo, X, y, cv=5)
gap = tr_sc.mean(1)[-1] - te_sc.mean(1)[-1]  # grande → overfitting

# C7: Cross-Validation y GridSearch
scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
gs = GridSearchCV(modelo, param_grid, cv=5, refit=True)

# C8: Clustering
X_sc = StandardScaler().fit_transform(X)
km   = KMeans(n_clusters=4, n_init=20, random_state=42)
sc   = silhouette_score(X_sc, km.fit_predict(X_sc))

# C9: Pipeline completo
pipe = Pipeline([('prepro', ct), ('clf', RandomForestClassifier())])
gs   = GridSearchCV(pipe, {'clf__n_estimators': [50,100]}, cv=5)
joblib.dump(gs.best_estimator_, 'modelo.joblib')
modelo = joblib.load('modelo.joblib')

# C10: Proyecto end-to-end
#  EDA → Feature Eng → Pipeline → Comparativa → GridSearch → Diagnóstico → Guardar
```

---

## ¡Enhorabuena!

Has completado el **Módulo 7 — Machine Learning**.

**El siguiente módulo** entrará en el mundo del **Deep Learning**: redes neuronales, PyTorch y las bases del aprendizaje profundo que potencia la IA moderna.